# Geometry reliability: Failure decomposition · Independent score models · Multi-seed ablation · Sample-only proxy

**Follow-up to Experiments 1–3; geometry estimation only.** This is a standalone Colab notebook. It addresses four explicit limitations:

1. **Failure attribution:** separate estimated-rank mistakes, learned-score errors, and Gaussian-smoothing bias on a synthetic torus with known tangent geometry. Use a numerical Gaussian-convolution reference rather than assuming the learned network equals the true score.
2. **Independent-model validation:** test candidate projector geometry using an independently trained score model; assess whether cross-model consistency predicts true tangent errors.
3. **Independent training seeds:** train or reload multiple independent score networks per manifold and evaluate an identical ablation on points withheld from the diagnostic fitting procedure.
4. **Sample-only neighborhood proxy:** use only coordinates of an independent point cloud, never synthetic analytic neighbor angles or true tangent labels, to construct the proxy.

**Do not overclaim:** Numerical quadrature is approximate and must pass grid-convergence checks. All true-error supervision in the synthetic regression is a *diagnostic experiment*, not a label-free calibration guarantee. A second score model might share bias with the first. The geometry estimator's spectral head matches the official LHSD Soft-DoF functional form in 8D, not the full official LHSD stochastic/scale-selection pipeline.

**Primary outputs:** `TORUS_FAILURE_DECOMPOSITION.csv`, `INDEPENDENT_MODEL_DIAGNOSTICS.csv`, `MULTISEED_ABLATION.csv`, `SAMPLE_ONLY_PROXY.csv`, `METHOD_AUDIT.json`, and figures. Reuses original checkpoints from `MyDrive/synthetic_tangent_ground_truth` if present. New checkpoints are stored separately and never overwrite the old ones.

## 0. Setup and frozen experiment design

In [ ]:
%pip -q install numpy pandas scipy scikit-learn matplotlib torch

In [ ]:
import os, time, json, math, random, hashlib, warnings
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.special import logsumexp
from scipy.stats import spearmanr
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import Ridge
from sklearn.metrics import roc_auc_score, mean_absolute_error
from IPython.display import display
import torch
from torch import nn

# ----- USER SETTINGS -----
FAMILIES = ['circle','sphere','ellipse','torus']  # For quick pilot, use ['circle','torus']
NEW_MODEL_SEEDS = [1701, 1702, 1703]          # Independent neural-network training runs
TRAIN_STEPS = 2500                            # Match original DSM training duration
TRAIN_BATCH = 512
N_FIT, N_CAL, N_TEST = 80, 80, 250           # Distinct points; same for all model seeds
N_REFERENCE = 6000                           # Independent coordinates for data-only kNN
K_NEIGHBORS = 12
SIGMAS = [0.030,0.128,0.342]
REFERENCE_SIGMA = {0.030:0.128,0.128:0.342,0.342:0.128}
QUADRATURE_RES = 256                        # Torus: 256² points; compare grid 256² vs 512²
RUN_NUMERICAL_REFERENCE = True
GRID_FINE = 512
GRID_COARSE = 256
GRID_TOL = 0.06                             # Disclose non-convergence; never silently pass it
N_GRID_AUDIT = 20
N_BOOT = 1000
LHSD_SOFT_C, LHSD_SOFT_P = 0.5, 12
H_FD = 0.001
SEED_GEOMETRY = 73                         # MUST match original synthetic geometry
SEED_DATA = 20261107                        # Fresh evaluation points (distinct from old tests)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

try:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = Path('/content/drive/MyDrive')
except ImportError:
    ROOT = Path('/mnt/data') if Path('/mnt/data').exists() else Path.cwd()
OLD_DATA = ROOT/'synthetic_tangent_ground_truth'
OLD_RESULTS = ROOT/'Orientation_Reliability_Exp123'
OUT = ROOT/'Geometry_Reliability_Followup_Independent_Seeds'
CKPT = OUT/'checkpoints'; CKPT.mkdir(parents=True,exist_ok=True)
OUT.mkdir(parents=True,exist_ok=True)

random.seed(SEED_DATA); np.random.seed(SEED_DATA); torch.manual_seed(SEED_DATA)
torch.set_num_threads(min(4,os.cpu_count() or 2))
print('Device:',DEVICE, '| Original checkpoints:',OLD_DATA,'| New output:',OUT)
print('Manifolds:', FAMILIES, ' | Training seeds:',NEW_MODEL_SEEDS)

## 0A. Reuse previous evidence (retrospective only)

This cell reads the earlier per-point CSV *if present* and audits the observed torus rank/orientation failure without retraining. It is descriptive and **never enters fitting or selecting the new models**.

In [ ]:
prior=OLD_RESULTS/'EXPERIMENT3_RIDGE_FRESH_PREDICTIONS.csv'
if not prior.exists():
    prior=ROOT/'EXPERIMENT3_RIDGE_FRESH_PREDICTIONS.csv'
if prior.exists():
    prev=pd.read_csv(prior)
    need={'family','sigma','estimated_rank','dim_true','tangent_error','estimated_rank_tangent_error'}
    assert need<=set(prev.columns),f'Missing required prior columns: {need-set(prev.columns)}'
    previous_torus=prev[prev.family=='torus'].copy()
    if not previous_torus.empty:
        previous_torus['correct_rank']=(previous_torus.estimated_rank==previous_torus.dim_true)
        audit=previous_torus.groupby('sigma').agg(
            points=('index','nunique'),rank_match=('correct_rank','mean'),
            oracle_rank_error=('tangent_error','mean'),
            estimated_rank_error=('estimated_rank_tangent_error','mean')).reset_index()
        audit.to_csv(OUT/'PRIOR_TORUS_AUDIT_DESCRIPTIVE.csv',index=False)
        print('Previously observed torus failure (DO NOT use these rows to tune):')
        display(audit.round(4))
else:
    print('Prior per-point CSV not present in Drive; new experiments work without it.')

### Experimental protocol and safeguards

- **Model seeds:** train separate networks with distinct RNG seeds; keep the data distribution, architecture, training steps, and optimizer fixed. Do not cherry-pick model seeds.
- **Point splits:** generated from disjoint RNG streams. The sample-only reference pool is independent of FIT, CAL, and TEST. All models evaluate the same points for paired comparison.
- **Score model:** original synthetic DSM epsilon model with \(Q_\sigma =\sigma J_{\epsilon_\theta}\). Do **not** identify this with the exact Hessian of a conservative energy unless conservativity is verified.
- **Torus reference:** for the same uniform-parameter torus distribution used in training, numerically compute \(\mu_\sigma(y)=E[X\mid X+\sigma Z=y]\) and \(Q_\sigma^{\rm ref}=I-\mathrm{Cov}[X\mid y]/\sigma^2\). Grid convergence is checked.
- **Fits:** fit supervised error predictors only with synthetic FIT labels, define exploratory failure thresholds on CAL, evaluate on untouched TEST. Measure point bootstrap **within** each model, and report variation across model seeds. This is not a true-error guarantee on MNIST.

## 1. The original score-network architecture and the original synthetic manifolds

In [ ]:
D=8
class EpsNetwork(nn.Module):
    def __init__(self,D=8,width=128):
        super().__init__()
        self.net=nn.Sequential(nn.Linear(D+1,width),nn.SiLU(),nn.Linear(width,width),nn.SiLU(),
                               nn.Linear(width,width),nn.SiLU(),nn.Linear(width,D))
    def forward(self,y,sigma):
        if not isinstance(sigma,torch.Tensor):
            sigma=torch.full((len(y),1),float(sigma),device=y.device,dtype=y.dtype)
        if sigma.ndim==1:sigma=sigma[:,None]
        return self.net(torch.cat([y,torch.log(sigma)],dim=1))

b0=np.random.default_rng(SEED_GEOMETRY).normal(size=(D,2)); B,_=np.linalg.qr(b0)
v0=np.random.default_rng(SEED_GEOMETRY+101).normal(size=D)
v0=v0-B@(B.T@v0);v0/=np.linalg.norm(v0)
B3=np.column_stack([B,v0])
A_ELL,B_ELL=1.30,0.70
R_MAJOR,R_MINOR=1.20,0.45

def sample_params(family,n,rng):
    if family in ('circle','ellipse'):return rng.uniform(0,2*np.pi,size=(n,1))
    if family=='sphere':return np.column_stack([rng.uniform(0,2*np.pi,n),rng.uniform(-1,1,n)])
    if family=='torus':return rng.uniform(0,2*np.pi,size=(n,2))
    raise ValueError(family)

def points_and_tangents(family,params):
    p=np.asarray(params,dtype=np.float64)
    if family=='circle':
        th=p[:,0]; x=np.cos(th)[:,None]*B[:,0]+np.sin(th)[:,None]*B[:,1]
        t=(-np.sin(th)[:,None]*B[:,0]+np.cos(th)[:,None]*B[:,1])[:,None,:]
        return x,t
    if family=='ellipse':
        th=p[:,0];x=np.stack([A_ELL*np.cos(th),B_ELL*np.sin(th)],axis=-1)@B.T
        v=(-A_ELL*np.sin(th))[:,None]*B[:,0]+(B_ELL*np.cos(th))[:,None]*B[:,1]
        return x,(v/np.linalg.norm(v,axis=1,keepdims=True))[:,None,:]
    if family=='sphere':
        th,z=p[:,0],p[:,1];a=np.sqrt(np.clip(1-z*z,1e-12,None))
        x=np.stack([a*np.cos(th),a*np.sin(th),z],axis=1)@B3.T
        t1=np.stack([-np.sin(th),np.cos(th),np.zeros_like(th)],axis=1)@B3.T
        t2=np.stack([-z*np.cos(th),-z*np.sin(th),a],axis=1)@B3.T
        return x,np.stack([t1,t2],axis=1)
    if family=='torus':
        u,v=p[:,0],p[:,1];radial=R_MAJOR+R_MINOR*np.cos(v)
        x=np.stack([radial*np.cos(u),radial*np.sin(u),R_MINOR*np.sin(v)],axis=1)@B3.T
        t1=np.stack([-np.sin(u),np.cos(u),np.zeros_like(u)],axis=1)@B3.T
        t2=np.stack([-np.sin(v)*np.cos(u),-np.sin(v)*np.sin(u),np.cos(v)],axis=1)@B3.T
        return x,np.stack([t1,t2],axis=1)
    raise ValueError(family)

def sample_points(family,n,seed):
    prm=sample_params(family,n,np.random.default_rng(seed))
    x,t=points_and_tangents(family,prm)
    return x.astype(np.float64), t.astype(np.float64),prm

# Direct unit tests: orthonormal tangents and orthogonal projection.
for fam in ['circle','sphere','ellipse','torus']:
    x,t,_=sample_points(fam,13,422)
    assert np.max(np.abs(t@np.swapaxes(t,1,2)-np.eye(t.shape[1])[None,:,:]))<1e-10
    assert x.shape==(13,D)
print('Synthetic geometry and tangent orthonormality: PASS')

## 2. Train / reload independent score models (no modification to pretrained original checkpoints)

In [ ]:
def sample_clean_torch(family,n,device):
    if family in ('circle','ellipse'):
        a=2*math.pi*torch.rand(n,device=device)
        scales=(1.,1.) if family=='circle' else (A_ELL,B_ELL)
        z=torch.stack([scales[0]*torch.cos(a),scales[1]*torch.sin(a)],1)
        return z@torch.as_tensor(B.T,dtype=torch.float32,device=device)
    if family=='sphere':
        a=2*math.pi*torch.rand(n,device=device);z=2*torch.rand(n,device=device)-1
        r=torch.sqrt(torch.clamp(1-z*z,min=0))
        v=torch.stack([r*torch.cos(a),r*torch.sin(a),z],1)
        return v@torch.as_tensor(B3.T,dtype=torch.float32,device=device)
    if family=='torus':
        u=2*math.pi*torch.rand(n,device=device);v=2*math.pi*torch.rand(n,device=device)
        a=R_MAJOR+R_MINOR*torch.cos(v)
        xyz=torch.stack([a*torch.cos(u),a*torch.sin(u),R_MINOR*torch.sin(v)],1)
        return xyz@torch.as_tensor(B3.T,dtype=torch.float32,device=device)
    raise ValueError(family)

def new_ckpt_file(family,seed):
    return CKPT/f'{family}_D8_seed{seed}_steps{TRAIN_STEPS}_net_v1.pt'

def load_or_train(family,seed):
    path=new_ckpt_file(family,seed)
    model=EpsNetwork(D).to(DEVICE)
    if path.exists():
        payload=torch.load(path,map_location=DEVICE,weights_only=True)
        assert payload['seed']==seed and payload['family']==family and payload['steps']==TRAIN_STEPS
        model.load_state_dict(payload['state']);print('Reused:',path.name)
    else:
        torch.manual_seed(seed+10000*['circle','sphere','ellipse','torus'].index(family))
        model=EpsNetwork(D).to(DEVICE)
        opt=torch.optim.Adam(model.parameters(),lr=1e-3)
        for step in range(TRAIN_STEPS):
            j=torch.randint(0,len(SIGMAS),(TRAIN_BATCH,),device=DEVICE)
            sig=torch.as_tensor(SIGMAS,dtype=torch.float32,device=DEVICE)[j,None]
            clean=sample_clean_torch(family,TRAIN_BATCH,DEVICE)
            eps=torch.randn_like(clean)
            pred=model(clean+sig*eps,sig)
            loss=(pred-eps).square().mean()
            opt.zero_grad();loss.backward();opt.step()
            if step in (0,TRAIN_STEPS-1) or (step+1)%500==0:
                print(f'{family}/{seed}: step {step+1}/{TRAIN_STEPS}, DSM loss {loss.item():.4f}')
        torch.save(dict(family=family,seed=seed,steps=TRAIN_STEPS,
                        architecture='EpsNetwork_8x128x128x128x8',state=model.state_dict()),path)
    model.eval()
    for par in model.parameters():par.requires_grad_(False)
    return model

def original_checkpoint(fam):
    if fam=='circle':return OLD_DATA/'synthetic_eps_network.pt'
    return OLD_DATA/f'{fam}_eps_R8_seed{SEED_GEOMETRY}_steps2500_v1.pt'

# Checkpoints are loaded during each family evaluation, avoiding 12 networks in RAM.
print('Original models available:', {f:original_checkpoint(f).exists() for f in FAMILIES})

## 3. Reference: the **Gaussian-smoothed true score**, not a perfect manifold projector

For \(Y=X+\sigma Z\), where \(Z\sim\mathcal N(0,I)\),
\[
\mu_\sigma(y)=\frac{\int x\exp(-\|y-x\|^2/(2\sigma^2))dP_X(x)}{\int\exp(-\|y-x\|^2/(2\sigma^2))dP_X(x)},\quad
Q^{\mathrm{ref}}_\sigma(y)=I-\frac{\operatorname{Cov}(X\mid Y=y)}{\sigma^2}.
\]
The second identity follows by differentiating the posterior mean. We approximate the integrals with deterministic quadrature over the **same distribution as the training data**. This distinguishes model approximation error \(\hat Q-Q^{\mathrm{ref}}\) from the smoothing-induced deviation of \(Q^{\mathrm{ref}}\)'s tangent eigenspace from the exact manifold tangent. Both differ from numerical finite-difference error.

An important subtlety: the torus training distribution samples **uniformly in its two angles**, not uniformly in surface area. The quadrature must preserve those weights.

In [ ]:
def quadrature_grid(fam,res):
    a=(np.arange(res)+0.5)/res
    if fam in ('circle','ellipse'):
        p=(2*np.pi*a)[:,None]
    elif fam=='sphere':
        aa,zz=np.meshgrid(2*np.pi*a,2*a-1,indexing='ij')
        p=np.stack([aa.ravel(),zz.ravel()],1)
    elif fam=='torus':
        aa,bb=np.meshgrid(2*np.pi*a,2*np.pi*a,indexing='ij')
        p=np.stack([aa.ravel(),bb.ravel()],1)
    else:raise ValueError(fam)
    X,_=points_and_tangents(fam,p)
    return np.asarray(X,dtype=np.float64)

def exact_smoothed_reference(family,Y,sigma,res,chunk=24):
    X=quadrature_grid(family,res)
    Y=np.asarray(Y,dtype=np.float64)
    gram=np.sum(X*X,axis=1)[None,:]
    QT=[]; mu_all=[]
    for start in range(0,len(Y),chunk):
        y=Y[start:start+chunk]
        dist2=np.maximum(np.sum(y*y,axis=1)[:,None]+gram-2*y@X.T,0.)
        logits=-dist2/(2*sigma*sigma)
        w=np.exp(logits-logsumexp(logits,axis=1,keepdims=True))
        mu=w@X
        XX=np.einsum('bm,mi,mj->bij',w,X,X,optimize=True)
        cov=XX-mu[:,:,None]*mu[:,None,:]
        Q=np.eye(D)[None,:,:]-cov/(sigma*sigma)
        QT.append(.5*(Q+np.swapaxes(Q,1,2)));mu_all.append(mu)
    return np.concatenate(QT),np.concatenate(mu_all)

# Closed-form Gaussian affine-subspace sanity check: if X~N(0,tau² I),
# posterior covariance=tau²*sigma²/(tau²+sigma²) I and Q=sigma²/(tau²+sigma²) I.
# This checks the identity mathematically, not quadrature implementation.
sig=.2;tau=.8
assert abs((1-(tau*tau*sig*sig/(tau*tau+sig*sig))/sig**2)-sig*sig/(tau*tau+sig*sig))<1e-12

# Convergence audit: compare coarse/fine quadrature on the SAME points for each sigma.
quad_audit=[]
if RUN_NUMERICAL_REFERENCE:
    for fam in FAMILIES:
        X,_,_=sample_points(fam,N_GRID_AUDIT,SEED_DATA+3000+['circle','sphere','ellipse','torus'].index(fam))
        for sig in SIGMAS:
            qc,_=exact_smoothed_reference(fam,X,sig,GRID_COARSE)
            qf,_=exact_smoothed_reference(fam,X,sig,GRID_FINE)
            rel=np.linalg.norm(qc-qf,axis=(1,2))/(np.linalg.norm(qf,axis=(1,2))+1e-12)
            quad_audit.append(dict(family=fam,sigma=sig,coarse=GRID_COARSE,fine=GRID_FINE,
                                   mean_relative_Q_grid_error=float(rel.mean()),
                                   max_relative_Q_grid_error=float(rel.max()),
                                   pass_tolerance=bool(rel.max()<GRID_TOL)))
    qa=pd.DataFrame(quad_audit)
    qa.to_csv(OUT/'NUMERICAL_QUADRATURE_CONVERGENCE.csv',index=False)
    display(qa.round(5))
    if not qa.pass_tolerance.all():
        print('WARNING: some reference grids did not converge. Increase QUADRATURE_RES and GRID_FINE; treat these rows as inconclusive.')
else:
    print('No reference run requested: learned-score-vs-exact-smoothing decomposition will be unavailable.')

## 4. Measure operators, geometry, observed-data proxies and independent responses

In [ ]:
@torch.no_grad()
def raw_q_fd(net,x,sigma,h=H_FD):
    x=np.asarray(x,dtype=np.float32)
    X=torch.from_numpy(x).to(DEVICE)
    outputs=[]
    for j in range(D):
        a=X.clone();z=X.clone();a[:,j]+=h;z[:,j]-=h
        q=(net(a,float(sigma))-net(z,float(sigma)))/(2*h)
        outputs.append(q.cpu().numpy())
    return float(sigma)*np.stack(outputs,axis=-1).astype(np.float64)

def projectors(Q,ranks):
    ev,vec=np.linalg.eigh(.5*(Q+np.swapaxes(Q,1,2)))
    p=np.zeros_like(Q)
    for i,r in enumerate(ranks):
        r=int(r)
        if r:p[i]=vec[i,:,:r]@vec[i,:,:r].T
    return p,ev

def soft_dof(Q):
    lam=np.linalg.eigvalsh(.5*(Q+np.swapaxes(Q,1,2)))
    return np.sum(1/(1+(np.maximum(lam,0)/LHSD_SOFT_C)**LHSD_SOFT_P),axis=1)

def fro_error(P,T):
    # Rank-independent denominator fixed to sqrt(true intrinsic dimension).
    return np.linalg.norm(P-T,axis=(1,2))/np.sqrt(np.maximum(1.,np.trace(T,axis1=1,axis2=2)))

# Reference is sample-only. NO true tangent, param, or analytic neighbors used.
def sample_only_proxy(Q,Y,reference,seed,k=K_NEIGHBORS):
    model=NearestNeighbors(n_neighbors=k,algorithm='auto').fit(reference)
    ix=model.kneighbors(Y,return_distance=False)
    delta=reference[ix]-Y[:,None,:]
    norm=np.linalg.norm(delta,axis=2,keepdims=True)
    d=delta/(norm+1e-12)
    rng=np.random.default_rng(seed)
    z=rng.normal(size=(len(Y),16,D));z/=np.linalg.norm(z,axis=2,keepdims=True)+1e-12
    e_neigh=np.mean(np.sum(np.einsum('nij,nkj->nki',Q,d)**2,axis=-1),axis=1)
    e_rand=np.mean(np.sum(np.einsum('nij,nkj->nki',Q,z)**2,axis=-1),axis=1)
    return np.minimum(1.,e_neigh/(e_rand+1e-12)), np.mean(norm[...,0],axis=1)

# No analytic tangent/parameter dependencies in the sample-only proxy function.
import inspect
assert all(bad not in inspect.signature(sample_only_proxy).parameters
           for bad in ['T','tangent','params','angles','truth'])
print('Matrix operator, tangent comparison and sample-only reference proxy: PASS')

## 5. Run original and independent score models on identically held-out points

In [ ]:
all_tables=[];run_metadata=[]
for fam_idx,fam in enumerate(FAMILIES):
    print('\n'+'='*68+'\nFAMILY:',fam,flush=True)
    prefix=OUT/f'{fam}_ALL_MODELS.csv'
    meta_path=OUT/f'{fam}_ALL_MODELS_PROTOCOL.json'
    family_protocol=dict(family=fam,TRAIN_STEPS=TRAIN_STEPS,model_seeds=NEW_MODEL_SEEDS,
        N_FIT=N_FIT,N_CAL=N_CAL,N_TEST=N_TEST,N_REFERENCE=N_REFERENCE,K_NEIGHBORS=K_NEIGHBORS,
        SIGMAS=SIGMAS,REFERENCE_SIGMA=REFERENCE_SIGMA,QUADRATURE_RES=QUADRATURE_RES,
        RUN_NUMERICAL_REFERENCE=RUN_NUMERICAL_REFERENCE,H_FD=H_FD,SEED_DATA=SEED_DATA,
        original_checkpoint_exists=original_checkpoint(fam).exists())
    if prefix.exists():
        if not meta_path.exists():
            raise RuntimeError(f'Found a cached table without protocol: {prefix}; move the table aside before rerunning')
        previous_protocol=json.loads(meta_path.read_text())
        assert previous_protocol==json.loads(json.dumps(family_protocol)),('Stale saved family table',fam)
        old=pd.read_csv(prefix)
        ids=set(f'seed_{s}' for s in NEW_MODEL_SEEDS)
        if original_checkpoint(fam).exists():ids.add('original')
        expected={'fit':N_FIT,'cal':N_CAL,'test':N_TEST}
        counts=old.groupby(['model_id','split']).point_id.nunique().to_dict()
        if (set(old.model_id)==ids and set(old.sigma)==set(SIGMAS)
            and all(counts.get((m,sp))==n for m in ids for sp,n in expected.items())
            and old['proxy_kind'].eq('independent_sample_only').all()):
            print('Reused completed family table:',prefix)
            all_tables.append(old);continue
        raise RuntimeError(f'Cached {prefix} has mismatched protocol. Move it aside before changing settings.')

    fam_seed=SEED_DATA+1000*fam_idx
    split_data={}
    for j,(name,count) in enumerate([('fit',N_FIT),('cal',N_CAL),('test',N_TEST)]):
        X,T,_=sample_points(fam,count,fam_seed+j)
        split_data[name]=(X,T)
    ref_X,_,_=sample_points(fam,N_REFERENCE,fam_seed+70)
    model_bank={}
    if original_checkpoint(fam).exists():
        a=EpsNetwork(D).to(DEVICE)
        a.load_state_dict(torch.load(original_checkpoint(fam),map_location=DEVICE,weights_only=True))
        a.eval();model_bank['original']=a
        print('Loaded original pretrained checkpoint for',fam)
    for seed in NEW_MODEL_SEEDS:
        model_bank[f'seed_{seed}']=load_or_train(fam,seed)
    model_ids=list(model_bank)
    assert len(model_ids)>=3,'Need at least 3 independently trained models for multi-seed assessment'
    print('Model bank:',model_ids)

    # Network tensors per split/model/scale; all independent peers see EXACTLY the same points.
    fam_rows=[]
    for split,(X,T) in split_data.items():
        n=len(X);true_rank=T.shape[1]
        Pt=np.einsum('nkd,nke->nde',T,T)
        ref_mats={};ref_mu={}
        if RUN_NUMERICAL_REFERENCE:
            for sig in SIGMAS:
                qr,mu=exact_smoothed_reference(fam,X,sig,QUADRATURE_RES)
                ref_mats[sig]=qr;ref_mu[sig]=mu
        bank_q={};bank_p={};bank_soft={};bank_rank={}
        for mid,net in model_bank.items():
            bank_q[mid]={};bank_p[mid]={};bank_rank[mid]={};bank_soft[mid]={}
            for sig in SIGMAS:
                qr=raw_q_fd(net,X,sig)
                qs=.5*(qr+np.swapaxes(qr,1,2))
                dof=soft_dof(qs)
                r=np.clip(np.rint(dof).astype(int),0,D)
                ph,_=projectors(qs,r)
                bank_q[mid][sig]=qr;bank_p[mid][sig]=ph
                bank_rank[mid][sig]=r;bank_soft[mid][sig]=dof
        for mid,net in model_bank.items():
            for sig in SIGMAS:
                q=bank_q[mid][sig];qs=.5*(q+np.swapaxes(q,1,2))
                p_est=bank_p[mid][sig];r=bank_rank[mid][sig]
                evals,vs=np.linalg.eigh(qs)
                p_oracle=vs[:,:,:true_rank]@np.swapaxes(vs[:,:,:true_rank],1,2)
                p_ref=None
                if RUN_NUMERICAL_REFERENCE:
                    q_ref=ref_mats[sig];ref_evs,ref_vs=np.linalg.eigh(q_ref)
                    p_ref=ref_vs[:,:,:true_rank]@np.swapaxes(ref_vs[:,:,:true_rank],1,2)
                # Peer model chosen *before* reading any measured errors.
                peer_id=model_ids[(model_ids.index(mid)+1)%len(model_ids)]
                peer_q=bank_q[peer_id][REFERENCE_SIGMA[sig]]
                own_ref_q=bank_q[mid][REFERENCE_SIGMA[sig]]
                A_peer=np.eye(D)[None,:,:]-peer_q
                A_own=np.eye(D)[None,:,:]-own_ref_q
                resp_ind=np.linalg.norm(A_peer-p_est,axis=(1,2))/np.sqrt(D)
                resp_same=np.linalg.norm(A_own-p_est,axis=(1,2))/np.sqrt(D)
                peer_p=bank_p[peer_id][sig]
                peerspace=np.linalg.norm(peer_p-p_est,axis=(1,2))/np.sqrt(D)
                scale_p=bank_p[mid][REFERENCE_SIGMA[sig]]
                scale_dis=np.linalg.norm(scale_p-p_est,axis=(1,2))/np.sqrt(D)
                q2=raw_q_fd(net,X,sig,h=2*H_FD)
                fd_rel=np.linalg.norm(q-q2,axis=(1,2))/(np.linalg.norm(qs,axis=(1,2))+1e-12)
                noncons=np.linalg.norm(q-np.swapaxes(q,1,2),axis=(1,2))/(np.linalg.norm(qs,axis=(1,2))+1e-12)
                defect=np.linalg.norm(qs@qs-qs,axis=(1,2))/(np.linalg.norm(qs,axis=(1,2))+1e-12)
                proxy,dist_ref=sample_only_proxy(qs,X,ref_X,seed=100+fam_idx,k=K_NEIGHBORS)
                e_est=fro_error(p_est,Pt);e_or=fro_error(p_oracle,Pt)
                e_rank=fro_error(p_est,p_oracle)
                e_mod=np.full(n,np.nan);e_sm=np.full(n,np.nan);qe=np.full(n,np.nan);eps_mse=np.full(n,np.nan)
                if RUN_NUMERICAL_REFERENCE:
                    e_mod=fro_error(p_oracle,p_ref)
                    e_sm=fro_error(p_ref,Pt)
                    qe=np.linalg.norm(qs-q_ref,axis=(1,2))/np.sqrt(D)
                    with torch.no_grad():
                        pred=net(torch.from_numpy(X.astype('float32')).to(DEVICE),sig).cpu().numpy()
                    eps_ref=(X-ref_mu[sig])/sig
                    eps_mse=np.mean((pred-eps_ref)**2,axis=1)
                    # Decomposition checks via exact triangle inequality.
                    assert np.all(e_est <= e_rank+e_mod+e_sm+1e-7),('triangular decomposition',fam,mid,sig)
                gap=evals[:,true_rank]-evals[:,true_rank-1] if true_rank<D else np.full(n,np.nan)
                for ix in range(n):
                    fam_rows.append(dict(family=fam,model_id=mid,peer_id=peer_id,split=split,point_id=ix,
                        sigma=sig,reference_sigma=REFERENCE_SIGMA[sig],true_rank=true_rank,
                        soft_dimension=float(bank_soft[mid][sig][ix]),estimated_rank=int(r[ix]),
                        dimension_MAE=float(abs(bank_soft[mid][sig][ix]-true_rank)),
                        estimated_rank_tangent_error=float(e_est[ix]),oracle_rank_tangent_error=float(e_or[ix]),
                        rank_component=float(e_rank[ix]),score_eigenspace_component=float(e_mod[ix]),
                        smoothing_component=float(e_sm[ix]),score_operator_error=float(qe[ix]),
                        epsilon_posterior_MSE=float(eps_mse[ix]),true_rank_eigengap=float(gap[ix]),
                        sample_knn_proxy=float(proxy[ix]),mean_knn_distance=float(dist_ref[ix]),
                        projector_defect=float(defect[ix]),nonconservativity=float(noncons[ix]),
                        numerical_fd_disagreement=float(fd_rel[ix]),same_model_response=float(resp_same[ix]),
                        independent_model_response=float(resp_ind[ix]),
                        cross_model_projector_disagreement=float(peerspace[ix]),
                        scale_projector_disagreement=float(scale_dis[ix]),
                        proxy_kind='independent_sample_only'))
        print(f'{fam} | split {split:4s} | {len(model_ids)} models × {len(SIGMAS)} scales × {n} points')
    out_fam=pd.DataFrame(fam_rows)
    assert not out_fam.duplicated(['model_id','split','point_id','sigma']).any()
    out_fam.to_csv(prefix,index=False)
    meta_path.write_text(json.dumps(family_protocol,indent=2))
    all_tables.append(out_fam)
    run_metadata.append(dict(family=fam,models=model_ids,checkpoint_paths=[str(new_ckpt_file(fam,s)) for s in NEW_MODEL_SEEDS]))
    del model_bank
    if DEVICE=='cuda':torch.cuda.empty_cache()

FULL=pd.concat(all_tables,ignore_index=True)
FULL.to_csv(OUT/'FULL_DIAGNOSTIC_TABLE.csv',index=False)
print('Completed:',len(FULL),'candidate rows across',FULL.family.nunique(),'manifolds')
display(FULL[ FULL.split=='test' ].groupby(['family','sigma']).agg(
    correct_rank_fraction=('estimated_rank',lambda z:np.mean(z==FULL[FULL.index.isin(z.index)]['true_rank'])),
    mean_true_error=('estimated_rank_tangent_error','mean'),
    mean_oracle_rank_error=('oracle_rank_tangent_error','mean')).round(4))

## 6. Failure mechanism: distinguish rank, model, smoothing and numerical errors

In [ ]:
MECH=(FULL[ FULL.split=='test' ].groupby(['family','model_id','sigma']).agg(
    n=('point_id','size'),rank_match=('estimated_rank',lambda x: float(np.mean(x.to_numpy()==FULL.loc[x.index,'true_rank'].to_numpy()))),
    estimated_error=('estimated_rank_tangent_error','mean'),oracle_rank_error=('oracle_rank_tangent_error','mean'),
    rank_component=('rank_component','mean'),score_eigenspace_component=('score_eigenspace_component','mean'),
    smoothing_component=('smoothing_component','mean'),score_operator_error=('score_operator_error','mean'),
    epsilon_posterior_MSE=('epsilon_posterior_MSE','mean'),
    numerical_fd_disagreement=('numerical_fd_disagreement','mean'),
    nonconservativity=('nonconservativity','mean')).reset_index())
MECH.to_csv(OUT/'TORUS_FAILURE_DECOMPOSITION.csv',index=False)
if 'torus' in set(MECH.family):
    display(MECH[MECH.family=='torus'].round(4))
# Score error vs 1st/2nd order effects is descriptive, not causal isolation.
fig,axs=plt.subplots(1,len(FAMILIES),figsize=(5.4*len(FAMILIES),4),squeeze=False)
for i,fam in enumerate(FAMILIES):
    ax=axs[0,i];z=MECH[MECH.family==fam]
    for mid,g in z.groupby('model_id'):
        ax.plot(g.sigma,g.oracle_rank_error,'o-',label=mid)
    ax.set(title=f'{fam}: oracle-rank tangent error',xlabel='Noise scale',ylabel='Mean true error')
    ax.legend(fontsize=7);ax.grid(alpha=.2)
fig.tight_layout();fig.savefig(OUT/'Fig_Failure_By_Seed_And_Scale.png',dpi=180,bbox_inches='tight');plt.show()
print('Note: e_total <= e_rank + e_model_reference + e_smoothing; components are upper-bound terms, not additive fractions.')

## 7. Independent model, sample-only proxy and uncertainty evidence

In [ ]:
from scipy.stats import spearmanr
IND=[]
for (fam,mid,sig),g in FULL[FULL.split=='test'].groupby(['family','model_id','sigma']):
    for target in ['oracle_rank_tangent_error','estimated_rank_tangent_error']:
        for feature in ['projector_defect','sample_knn_proxy','same_model_response','independent_model_response',
                        'cross_model_projector_disagreement','scale_projector_disagreement',
                        'numerical_fd_disagreement','nonconservativity']:
            rho,_=spearmanr(g[feature],g[target])
            IND.append(dict(family=fam,model_id=mid,sigma=sig,target=target,feature=feature,
                            spearman_rho=float(rho),n_points=len(g)))
IND=pd.DataFrame(IND)
IND.to_csv(OUT/'INDEPENDENT_MODEL_DIAGNOSTICS.csv',index=False)
print('Across-model per-family Spearman summary (sign may change across sigmas):')
display(IND[(IND.target=='oracle_rank_tangent_error')].groupby(['family','feature']).spearman_rho.agg(['mean','min','max']).round(3))
PROXY=FULL[FULL.split=='test'].groupby(['family','model_id','sigma']).agg(
    sample_knn_proxy=('sample_knn_proxy','mean'),
    mean_knn_distance=('mean_knn_distance','mean'),
    true_tangent_error=('oracle_rank_tangent_error','mean'),
    estimated_rank_error=('estimated_rank_tangent_error','mean')).reset_index()
PROXY.to_csv(OUT/'SAMPLE_ONLY_PROXY.csv',index=False)

fig,axs=plt.subplots(1,2,figsize=(11,4))
for fam,g in FULL[FULL.split=='test'].groupby('family'):
    axs[0].scatter(g.sample_knn_proxy,g.oracle_rank_tangent_error,alpha=.15,s=8,label=fam)
    axs[1].scatter(g.independent_model_response,g.oracle_rank_tangent_error,alpha=.15,s=8,label=fam)
axs[0].set(xlabel='Sample-only kNN proxy',ylabel='True tangent error',title='Observational sample-only proxy')
axs[1].set(xlabel='Independent-score response',ylabel='True tangent error',title='Model-independent response check')
for ax in axs:ax.grid(alpha=.2);ax.legend(fontsize=8)
fig.tight_layout();fig.savefig(OUT/'Fig_Independent_Evidence.png',dpi=180,bbox_inches='tight');plt.show()

## 8. Leave-one-model-seed-out ablation (same model class, same point splits)

All regression input features below are computed without analytic tangent labels. Synthetic tangent labels are used only to **fit and evaluate the error predictors**. A held-out **model seed** has never supplied training labels; its **TEST points** are also independent. Include the pretrained original model as an additional fold if a checkpoint exists.

Compare a frozen ridge model with the same supervised FIT labels in each setting. Do not tune based on TEST. Thresholds for exploratory failure AUROC are fixed from other-model CAL points (75th percentile). Report results **per held-out network seed**; with three seeds, do not claim population-level confidence from point bootstraps.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
BASE=['sigma','soft_dimension','sample_knn_proxy','projector_defect',
      'scale_projector_disagreement','numerical_fd_disagreement','nonconservativity']
ABLATIONS={
    'base':BASE,
    'base+same_response':BASE+['same_model_response'],
    'base+independent_response':BASE+['independent_model_response'],
    'base+model_agreement':BASE+['cross_model_projector_disagreement'],
    'base+both_independent':BASE+['independent_model_response','cross_model_projector_disagreement'],
}
REGRESSORS=['ridge','random_forest']
def new_regressor(reg):
    if reg=='ridge':return make_pipeline(StandardScaler(),Ridge(alpha=3.0))
    if reg=='random_forest':return RandomForestRegressor(n_estimators=120,max_depth=5,min_samples_leaf=3,
                                                       random_state=SEED_DATA,n_jobs=-1)
    raise ValueError(reg)
ABL=[];PREDS=[]
for fam,z in FULL.groupby('family'):
    model_ids=sorted(z.model_id.unique())
    assert len(model_ids)>=3
    for held in model_ids:
        fit=z[(z.split=='fit')&(z.model_id!=held)]
        cal=z[(z.split=='cal')&(z.model_id!=held)]
        test=z[(z.split=='test')&(z.model_id==held)].copy()
        threshold=float(cal.estimated_rank_tangent_error.quantile(.75))
        failure=(test.estimated_rank_tangent_error.to_numpy()>threshold).astype(int)
        for reg in REGRESSORS:
            for name,keys in ABLATIONS.items():
                assert fit[keys].notna().all().all() and test[keys].notna().all().all()
                estimator=new_regressor(reg)
                estimator.fit(fit[keys],fit.estimated_rank_tangent_error)
                pred=estimator.predict(test[keys])
                absolute_error=np.abs(pred-test.estimated_rank_tangent_error.to_numpy())
                auc=(float(roc_auc_score(failure,pred)) if len(np.unique(failure))==2 else float('nan'))
                tmp=test[['point_id','estimated_rank_tangent_error']].copy();tmp['prediction']=pred
                ranked=tmp.groupby('point_id').agg(error=('estimated_rank_tangent_error','mean'),
                    risk=('prediction','mean')).sort_values('risk')
                k=max(1,int(np.ceil(.8*len(ranked))))
                ABL.append(dict(family=fam,heldout_model=held,regressor=reg,method=name,
                    n_train_models=len(model_ids)-1,n_fit_points_per_model=N_FIT,n_test_points=test.point_id.nunique(),
                    MAE=float(absolute_error.mean()),failure_threshold_from_CAL=threshold,
                    failure_prevalence=float(failure.mean()),failure_AUROC=auc,
                    mean_true_error_at_80pct_coverage=float(ranked.iloc[:k].error.mean())))
                tmp2=test[['point_id','sigma','estimated_rank_tangent_error']].copy()
                tmp2['family']=fam;tmp2['heldout_model']=held;tmp2['regressor']=reg
                tmp2['method']=name;tmp2['prediction']=pred;tmp2['absolute_error']=absolute_error
                PREDS.append(tmp2)
ABL=pd.DataFrame(ABL);PREDS=pd.concat(PREDS,ignore_index=True)
ABL.to_csv(OUT/'MULTISEED_ABLATION.csv',index=False)
PREDS.to_csv(OUT/'MULTISEED_ABLATION_PREDICTIONS.csv',index=False)
w=ABL.pivot_table(index=['family','heldout_model','regressor'],columns='method',values='MAE').reset_index()
for k in [name for name in ABLATIONS if name!='base']:
    w['base_minus_'+k]=w['base']-w[k]
w.to_csv(OUT/'MULTISEED_PAIRED_DIFFERENCES.csv',index=False)
print('Paired comparisons: each held-out model seed is an independent training replication.')
display(w.round(5))
cols=[k for k in w.columns if k.startswith('base_minus_')]
display(w.groupby(['family','regressor'])[cols].agg(['mean','min','max']).round(5))
# Primary prespecified ablation: independent response versus baseline.
PB=[]
for (fam,held,reg),g in PREDS.groupby(['family','heldout_model','regressor']):
    a=g[g.method=='base'].set_index(['point_id','sigma']).absolute_error
    q=g[g.method=='base+independent_response'].set_index(['point_id','sigma']).absolute_error
    diff=(a-q).groupby(level='point_id').mean().to_numpy()
    rng=np.random.default_rng(SEED_DATA+sum(map(ord,fam+held+reg)))
    bs=np.mean(diff[rng.integers(0,len(diff),size=(N_BOOT,len(diff)))],axis=1)
    lo,hi=np.quantile(bs,[.025,.975])
    PB.append(dict(family=fam,heldout_model=held,regressor=reg,delta_mean=float(diff.mean()),
        CI95_low=float(lo),CI95_high=float(hi),
        inference_scope='point-level conditional on heldout network (not a model-seed population CI)'))
PB=pd.DataFrame(PB);PB.to_csv(OUT/'MULTISEED_POINT_BOOTSTRAP.csv',index=False)
display(PB.round(6))
fig,axs=plt.subplots(1,len(FAMILIES),figsize=(5.1*len(FAMILIES),4),squeeze=False)
for j,fam in enumerate(FAMILIES):
    ax=axs[0,j]
    for reg,g in w[w.family==fam].groupby('regressor'):
        ax.plot(g.heldout_model,g['base_minus_base+independent_response'],'o-',label=reg)
    ax.axhline(0,color='k',lw=1)
    ax.set(title=fam,ylabel='MAE gain from independent model response')
    ax.tick_params(axis='x',rotation=35,labelsize=8);ax.legend(fontsize=8)
fig.tight_layout();fig.savefig(OUT/'Fig_Seed_Ablation.png',dpi=180,bbox_inches='tight');plt.show()

## 9. Numerical + validity audit, and proposal-ready report

In [ ]:
# Central reproducibility artifact: explicit protocol, caveats, and validation checks.
protocol={
    'research_question':'Can observable geometry diagnostics predict true tangent errors reliably across independent score networks?',
    'families':FAMILIES,'model_seeds':NEW_MODEL_SEEDS,'training_steps':TRAIN_STEPS,
    'data_seed':SEED_DATA,'training_architecture':'EpsNetwork(D=8,width=128)',
    'normalization':'Q=sigma J_epsilon; A=I-Q; pixel/ambient coordinates R8',
    'splits':{'fit':N_FIT,'cal':N_CAL,'test':N_TEST,'independent_reference':N_REFERENCE},
    'noise_scales':SIGMAS,'reference_sigma':REFERENCE_SIGMA,'quadrature_resolution':QUADRATURE_RES,
    'grid_coarse':GRID_COARSE,'grid_fine':GRID_FINE,
    'grid_reference_converged': bool(pd.DataFrame(quad_audit).pass_tolerance.all()) if quad_audit else None,
    'nearest_neighbor_reference_type':'independent sample cloud; NO analytic parameter/tangent labels',
    'rank_estimator':'rounded exact 8D LHSD Soft-DoF spectral head, fixed p and c',
    'tangent_loss':'normalized Frobenius projector error / sqrt(true intrinsic dimension)',
    'supervision':'true tangent labels permitted for synthetic FIT error-predictor training ONLY',
    'known_limits':['no label-free true-error guarantee on real datasets','same-family peer models may share bias',
       'Gaussian reference is numerical quadrature and resolution dependent',
       'leave-one-seed-out is not a population-level confidence guarantee with only three model seeds',
       'original pretrained model optional; source settings recorded separately',
       'official LHSD scale selection and SLQ implementation not reproduced'],
}
(OUT/'METHOD_AUDIT.json').write_text(json.dumps(protocol,indent=2,default=str))
mean0=ABL.groupby(['family','regressor','method']).MAE.mean().unstack()
report=['# Geometry reliability: reproducible follow-up evidence','',
    '## Design','Multiple DSM score-model training seeds; disjoint FIT/CAL/TEST points; sample-only kNN reference pool;',
    'numerical Gaussian smoothed-score reference; cross-network denoiser response; leave-one-network-out error prediction.','',
    '## Observations (descriptive, not proof of novelty or true-error guarantees)']
for fam in FAMILIES:
    t=MECH[MECH.family==fam]
    report.append(f'### {fam}')
    report.append(f'- Mean oracle-rank tangent error (across models and scales): {t.oracle_rank_error.mean():.4f}')
    report.append(f'- Correct estimated rank (mean across models and scales): {t.rank_match.mean():.3f}')
    if RUN_NUMERICAL_REFERENCE:
        report.append(f'- Score eigenspace discrepancy vs smoothed true-score reference: {t.score_eigenspace_component.mean():.4f}')
        report.append(f'- Smoothed-reference tangent discrepancy: {t.smoothing_component.mean():.4f}')
        aa=pd.DataFrame(quad_audit);aa=aa[aa.family==fam]
        report.append(f'- Reference quadrature max relative disagreement: {aa.max_relative_Q_grid_error.max():.4f}')
    for reg in REGRESSORS:
        imp=mean0.loc[(fam,reg)]
        report.append(f'- {reg} leave-one-network-out MAE base: {imp["base"]:.5f}; +independent response: {imp["base+independent_response"]:.5f}')
report.extend(['','## Mandatory interpretation',
    '- Model/reference/smoothing/rank terms are a **triangle inequality decomposition**, not additive causal percentages.',
    '- Fit error predictors require true synthetic tangent labels; the observable features themselves do not.',
    '- Grid-convergence failures invalidate a confident score-versus-smoothing attribution.',
    '- A positive point-bootstrap CI does not establish reproducibility over independent neural-model seeds.',
    '- The original EpsNetwork is not necessarily conservative; antisymmetry is measured.',
    '- Do not call this a full LHSD reproduction or a real-data true-tangent certificate.',
    '', '## Outputs',
    'TORUS_FAILURE_DECOMPOSITION.csv; INDEPENDENT_MODEL_DIAGNOSTICS.csv; MULTISEED_ABLATION.csv;',
    'SAMPLE_ONLY_PROXY.csv; NUMERICAL_QUADRATURE_CONVERGENCE.csv; MULTISEED_POINT_BOOTSTRAP.csv'])
(OUT/'PROPOSAL_EVIDENCE_FOLLOWUP.md').write_text('\n'.join(report))
print('Saved all results:',OUT)
print('Report preview:');print('\n'.join(report[:30]))

## 10. Archive and transfer

In [ ]:
import shutil
archive=shutil.make_archive(str(OUT.parent/'Geometry_Reliability_Followup_Independent_Seeds'),'zip',root_dir=str(OUT))
print('Archive:',archive)
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    print('Outside Colab; retrieve archive from the path above.')